In [0]:
## Initializatoin

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim

# Read Bronze Tabel

In [0]:
df = spark.read.table("workspace.bronze.erp_px_cat_g1v2_raw")

# Silver Transformations

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df.withColumn(field.name, trim(col(field.name)))

## Normalizing Maintenance Falg to Boolean

In [0]:
df = df.withColumn(
    "MAINTENANCE",
    F.when(F.upper(col("MAINTENANCE")).isin("Y", "YES"), F.lit(True))
    .when(F.upper(col("MAINTENANCE")).isin("N", "NO"), F.lit(False))
    .otherwise("n/a")
)

## Renaming Columns

In [0]:
RENAME_MAP = {
    "ID": "category_id",
    "CAT": "category",
    "SUBCAT": "sub_category",
    "MAINTENANCE": "maintenance_flag"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity Chack of DataFrame

In [0]:
display(df)

# Writting Silver Table

In [0]:
df.write.mode("overwrite").saveAsTable("workspace.silver.erp_product_category")

## Sanity Check of Silver Table

In [0]:
%sql

SELECT *
FROM workspace.silver.erp_product_category
LIMIT 10